In [ ]:
%%bash
D=/kaggle/input/notebooks/thunvng/hello-done/outputs/logs
echo "===== eval_dev.log (cuối)"
tr '\r' '\n' < $D/eval_dev.log | grep -v -i "warn\|it/s\]" | tail -40
echo; echo "===== loss SFT"
tr '\r' '\n' < $D/train.log | awk '/step examples/{f=1} f' | grep -E "step examples|'loss'|train_runtime" | (head -4; echo ...; tail -3)
echo; echo "===== loss ASR cuối"
tr '\r' '\n' < $D/train.log | awk '/step examples/{exit} {print}' | grep -E "'loss'|train_runtime" | tail -3


In [ ]:
%%bash
SRC=$(find /kaggle/input -name kaggle_setup.sh -print -quit)
[ -n "$SRC" ] || { echo "ERROR: không thấy kaggle_setup.sh, dataset code đã gắn chưa?"; exit 1; }
QW=$(grep -rlE --include=config.json '"model_type"[[:space:]]*:[[:space:]]*"qwen3"' /kaggle/input 2>/dev/null | head -1 | xargs -r dirname)
PH=$(grep -rlE --include=config.json '"model_type"[[:space:]]*:[[:space:]]*"whisper"' /kaggle/input 2>/dev/null | head -1 | xargs -r dirname)
echo "Qwen3 dir  : ${QW:-KHÔNG THẤY}"; echo "Whisper dir: ${PH:-KHÔNG THẤY}"
[ -n "$QW" ] && [ -n "$PH" ] || { echo "ERROR: thiếu thư mục model"; exit 1; }
cp "$SRC" /kaggle/working/setup_patched.sh
sed -i -e "s#^QWEN=.*#QWEN=$QW#" -e "s#^PHO=.*#PHO=$PH#" -e 's/python -I /python /g' /kaggle/working/setup_patched.sh
bash /kaggle/working/setup_patched.sh
sed -i 's/python -I /python /g' /kaggle/working/lifemate/scripts/*.sh
echo "--- đã khôi phục?"; ls /kaggle/working/checkpoints /kaggle/working/checkpoints/q4b_lora | head


In [ ]:
%%bash
source /kaggle/working/lifemate/kaggle_env.sh
ls -la outputs/*.jsonl 2>/dev/null
for f in outputs/dev_sft_*.jsonl; do [ -s "$f" ] && { echo "=== $f"; python -X utf8 scripts/eval.py --sub "$f" --ids data/processed/dev_ids.txt | tail -8; }; done


In [ ]:
%%bash
pip uninstall -y torchao 2>&1 | tail -2
python -c "import peft, transformers; print('peft', peft.__version__, '| transformers', transformers.__version__, '| import OK')"


In [ ]:
%%bash
source /kaggle/working/lifemate/kaggle_env.sh
# (a) ASR đã fine-tune: kịch bản gần bài nộp nhất
TAG=phowl_ft FORCE=1 bash scripts/kaggle_pipeline.sh eval_dev
# (b) trần: transcript sạch
bash scripts/infer_multi_gpu.sh train outputs/dev_sft_clean.jsonl --ids data/processed/dev_ids.txt --backend hf --model $QWEN --adapter checkpoints/q4b_lora --text clean
python -X utf8 scripts/eval.py --sub outputs/dev_sft_clean.jsonl --ids data/processed/dev_ids.txt
tail -qn 1 outputs/dev_sft_phowl_ft.part*.log outputs/dev_sft_clean.part*.log


In [ ]:
%%bash
source /kaggle/working/lifemate/kaggle_env.sh
bash scripts/infer_multi_gpu.sh train outputs/dev_q4b_clean.jsonl --ids data/processed/dev_ids.txt --backend hf --model $QWEN --text clean
python -X utf8 scripts/eval.py --sub outputs/dev_q4b_clean.jsonl --ids data/processed/dev_ids.txt
tail -qn 1 outputs/dev_q4b_clean.part*.log


In [ ]:
%%bash
source /kaggle/working/lifemate/kaggle_env.sh
O=/kaggle/working/deliverables; rm -rf $O
mkdir -p $O/q4b_lora $O/phowl_ft_merged $O/logs $O/dev_outputs
cp checkpoints/q4b_lora/{adapter_config.json,adapter_model.safetensors,tokenizer.json,tokenizer_config.json,chat_template.jinja} $O/q4b_lora/
cp -r checkpoints/phowl_ft/merged/. $O/phowl_ft_merged/
cp outputs/logs/*.log $O/logs/ 2>/dev/null
cp outputs/dev_*.jsonl $O/dev_outputs/ 2>/dev/null
( cd $O && find . -type f ! -name SHA256SUMS.txt -print0 | sort -z | xargs -0 sha256sum > SHA256SUMS.txt )
du -sh $O/*; echo; grep -E "adapter_model|safetensors" $O/SHA256SUMS.txt


In [ ]:
%%bash
bash /kaggle/working/lifemate/scripts/kaggle_save_state.sh
du -sh /kaggle/working/deliverables /kaggle/working/state /kaggle/working/checkpoints
